# Patch Extraction — 4-Tile Rebuild (36JTM, 36JUM, 36JTN, 36JUN)

**Project:** Deep Learning for Flood Inundation Mapping Using Multi-Source Satellite Data
**Study Area:** KwaZulu-Natal, South Africa (4-tile mosaic)
**Flood Event:** April 2022 KwaZulu-Natal Floods
**Author:** Valencia
**Supervisor:** Prof. Innocent Davidson
**Institution:** Cape Peninsula University of Technology (CPUT)

---

## Purpose of this Notebook

Extracts training patches from the rebuilt 4-tile stack and consolidated
UNOSAT flood label, following the same strategy as the original
`A1_full_tile_patch_extraction.ipynb`:

- Flood patches (containing at least 1 flood pixel) sampled densely
  (stride=32) from the flood-affected region
- Each flood patch augmented 8x (4 rotations x 2 flips)
- Non-flood patches sampled at a 10:1 ratio to flood patches
- Split: 70% train / 15% val / 15% test

**Important difference from the original pipeline:** the rebuilt stack
covers four tiles, of which only a small region (around the JTM/JUM
boundary) contains actual UNOSAT flood ground truth. The 36JTN tile
additionally has substantial Sentinel-1 nodata coverage (84.88%, accepted
as a known limitation since no flood polygons fall within it — see
`B3_mosaic.ipynb`). Non-flood patch sampling explicitly avoids
high-nodata regions, so training patches are not drawn from areas with
unreliable SAR data.

**Input:**
- `Stacked_4tile/flood_stack_4tile_v1.tif` — 12-band stack
- `Mosaic_4tile/flood_label_mosaic.tif` — rasterized UNOSAT label

**Output:** Individual patch `.npy` files in `Patches_4tile/{train,val,test}`

---
## Step 1: Mount Drive and Set Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!pip install rasterio --quiet

In [3]:
import os
import gc
import numpy as np
import rasterio
from rasterio.windows import Window
import warnings
warnings.filterwarnings('ignore')

ROOT       = '/content/drive/MyDrive/KZN_Research_Colab/'
STACK_PATH = ROOT + 'Stacked_4tile/flood_stack_4tile_v1.tif'
LABEL_PATH = ROOT + 'Mosaic_4tile/flood_label_mosaic.tif'
PATCH_DIR  = ROOT + 'Patches_4tile/'

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(PATCH_DIR, split), exist_ok=True)

PATCH_SIZE = 128
STRIDE     = 32     # dense extraction around flood region
MIN_FLOOD  = 1       # minimum flood pixels to count as flood patch
RATIO      = 10      # non-flood to flood patch ratio
MAX_NODATA_FRACTION = 0.3  # reject non-flood patches with more than this fraction of zero/nodata pixels

print('Paths and settings configured.')
print(f'  Stack      : {STACK_PATH}')
print(f'  Label      : {LABEL_PATH}')
print(f'  Output     : {PATCH_DIR}')
print(f'  Patch size : {PATCH_SIZE} x {PATCH_SIZE} pixels')
print(f'  Stride     : {STRIDE} pixels (flood region)')
print(f'  Flood:non-flood ratio : 1:{RATIO}')
print(f'  Max nodata fraction for non-flood patches: {MAX_NODATA_FRACTION}')

Paths and settings configured.
  Stack      : /content/drive/MyDrive/KZN_Research_Colab/Stacked_4tile/flood_stack_4tile_v1.tif
  Label      : /content/drive/MyDrive/KZN_Research_Colab/Mosaic_4tile/flood_label_mosaic.tif
  Output     : /content/drive/MyDrive/KZN_Research_Colab/Patches_4tile/
  Patch size : 128 x 128 pixels
  Stride     : 32 pixels (flood region)
  Flood:non-flood ratio : 1:10
  Max nodata fraction for non-flood patches: 0.3


---
## Step 2: Load the Flood Label and Identify the Flood Region

The label raster is loaded into memory (it's much smaller and simpler
than the 12-band stack, so this is safe). The flood region bounding box
is identified to focus dense patch extraction where flood pixels are
present — same approach as the original A1 notebook.

In [6]:
print('Loading flood label...')
with rasterio.open(LABEL_PATH) as src:
    label_full = src.read(1).astype(np.uint8)
    H, W = src.height, src.width
    label_transform = src.transform
    label_crs = src.crs

with rasterio.open(STACK_PATH) as src:
    stack_H, stack_W = src.height, src.width
    stack_transform = src.transform
    stack_crs = src.crs

print(f'  Label shape : {H} x {W}')
print(f'  Stack shape : {stack_H} x {stack_W}')
print(f'  Shapes match: {(H, W) == (stack_H, stack_W)}')

flood_px  = int((label_full == 1).sum())
flood_pct = 100 * flood_px / label_full.size
print(f'  Flood pixels: {flood_px:,}  ({flood_pct:.4f}%)')

rows_f, cols_f = np.where(label_full == 1)
row_min = max(0, rows_f.min() - 256)
row_max = min(H, rows_f.max() + 256)
col_min = max(0, cols_f.min() - 256)
col_max = min(W, cols_f.max() + 256)

flood_region_h = row_max - row_min
flood_region_w = col_max - col_min

print(f'\nFlood region (with 256px buffer):')
print(f'  Rows : {row_min} to {row_max}  ({flood_region_h} px = {flood_region_h*10/1000:.1f} km)')
print(f'  Cols : {col_min} to {col_max}  ({flood_region_w} px = {flood_region_w*10/1000:.1f} km)')

Loading flood label...
  Label shape : 22139 x 21576
  Stack shape : 22139 x 21576
  Shapes match: True
  Flood pixels: 98,715  (0.0207%)

Flood region (with 256px buffer):
  Rows : 9237 to 12364  (3127 px = 31.3 km)
  Cols : 8805 to 12208  (3403 px = 34.0 km)


---
## Step 3: Build a Nodata Mask for Non-Flood Patch Filtering

Since the 4-tile mosaic includes a region (36JTN) with substantial
Sentinel-1 nodata, a coarse nodata mask is built from the stack's
VV-post band so that non-flood patches are not drawn from areas with
unreliable SAR data. This check is done on a downsampled overview rather
than the full-resolution stack, to keep memory use low.

In [7]:
# Read VV_post (band 11) at a reduced resolution to build a coarse nodata mask
DOWNSAMPLE = 8  # read at 1/8 resolution for the mask only

with rasterio.open(STACK_PATH) as src:
    out_shape = (src.height // DOWNSAMPLE, src.width // DOWNSAMPLE)
    vv_post_small = src.read(11, out_shape=out_shape, resampling=rasterio.enums.Resampling.average)

# In the cleaned stack, nodata was zero-filled during assembly (see B4),
# so a value of exactly 0.0 indicates a nodata/seam pixel for this band
nodata_mask_small = (vv_post_small == 0.0)
nodata_pct = 100 * nodata_mask_small.sum() / nodata_mask_small.size
print(f'Coarse nodata mask built from VV_post (downsampled {DOWNSAMPLE}x).')
print(f'  Nodata fraction: {nodata_pct:.1f}%')

def nodata_fraction_at(row, col, size=PATCH_SIZE):
    """Estimate the nodata fraction for a patch location using the coarse mask."""
    r0, r1 = row // DOWNSAMPLE, (row + size) // DOWNSAMPLE
    c0, c1 = col // DOWNSAMPLE, (col + size) // DOWNSAMPLE
    r1 = max(r1, r0 + 1)
    c1 = max(c1, c0 + 1)
    patch_mask = nodata_mask_small[r0:r1, c0:c1]
    if patch_mask.size == 0:
        return 1.0
    return patch_mask.mean()

print('Nodata-checking function defined.')

Coarse nodata mask built from VV_post (downsampled 8x).
  Nodata fraction: 34.5%
Nodata-checking function defined.


---
## Step 4: Identify Flood and Non-Flood Patch Positions

Same strategy as the original A1 notebook: dense scan of the flood
region for flood patches, coarser scan of the full mosaic for non-flood
background patches. Non-flood candidates with excessive nodata are
filtered out using the mask built in Step 3.

In [8]:
flood_patches   = []
noflood_patches = []

print('Scanning flood region for patches...')
for row in range(row_min, row_max - PATCH_SIZE + 1, STRIDE):
    for col in range(col_min, col_max - PATCH_SIZE + 1, STRIDE):
        patch = label_full[row:row+PATCH_SIZE, col:col+PATCH_SIZE]
        if np.sum(patch == 1) >= MIN_FLOOD:
            flood_patches.append((row, col))
        else:
            if nodata_fraction_at(row, col) <= MAX_NODATA_FRACTION:
                noflood_patches.append((row, col))

print('Scanning full mosaic for additional non-flood background patches...')
for row in range(0, H - PATCH_SIZE + 1, 256):
    for col in range(0, W - PATCH_SIZE + 1, 256):
        patch = label_full[row:row+PATCH_SIZE, col:col+PATCH_SIZE]
        if np.sum(patch == 1) == 0:
            if nodata_fraction_at(row, col) <= MAX_NODATA_FRACTION:
                noflood_patches.append((row, col))

print(f'\nCandidate flood patches    : {len(flood_patches):,}')
print(f'Candidate non-flood patches: {len(noflood_patches):,}  (after nodata filtering)')

Scanning flood region for patches...
Scanning full mosaic for additional non-flood background patches...

Candidate flood patches    : 1,021
Candidate non-flood patches: 11,689  (after nodata filtering)


In [9]:
# Sanity check: do the flood patch positions actually cover the known flood pixel locations?
covered_flood_px = set()
for row, col in flood_patches:
    patch_label = label_full[row:row+PATCH_SIZE, col:col+PATCH_SIZE]
    ys, xs = np.where(patch_label == 1)
    for y, x in zip(ys, xs):
        covered_flood_px.add((row + y, col + x))

print(f'Unique flood pixels covered by flood patches: {len(covered_flood_px):,}')
print(f'Total flood pixels in label: 98,715')
print(f'Coverage: {100 * len(covered_flood_px) / 98715:.1f}%')

Unique flood pixels covered by flood patches: 98,715
Total flood pixels in label: 98,715
Coverage: 100.0%


In [10]:
np.random.seed(42)
n_noflood  = min(len(noflood_patches), len(flood_patches) * RATIO)
noflood_sel = [noflood_patches[i] for i in
               np.random.choice(len(noflood_patches), n_noflood, replace=False)]

all_patches = ([(r, c, True)  for r, c in flood_patches] +
               [(r, c, False) for r, c in noflood_sel])
np.random.shuffle(all_patches)

n       = len(all_patches)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

splits = {
    'train': all_patches[:n_train],
    'val'  : all_patches[n_train:n_train + n_val],
    'test' : all_patches[n_train + n_val:]
}

print(f'Patch positions identified:')
print(f'  Flood patches     : {len(flood_patches):,}')
print(f'  Non-flood patches : {n_noflood:,}  (sampled at {RATIO}:1 ratio)')
print(f'  Total positions   : {n:,}')
print(f'  Train/Val/Test    : {n_train}/{n_val}/{n-n_train-n_val}')
print(f'      Expected training patches after augmentation: ~{n_train + len([p for p in splits["train"] if p[2]]) * 7:,}')

Patch positions identified:
  Flood patches     : 1,021
  Non-flood patches : 10,210  (sampled at 10:1 ratio)
  Total positions   : 11,231
  Train/Val/Test    : 7861/1684/1686
      Expected training patches after augmentation: ~12,789


---
## Step 5: Define Augmentation Functions

Identical to the original A1 notebook: 8 geometric augmentations per
flood patch (4 rotations x 2 flips). Non-flood patches receive no
augmentation.

In [11]:
def augment(X, y, k):
    """
    X shape: (channels, H, W)
    y shape: (H, W)
    k: augmentation index 0-7
    """
    if k == 0: return X, y
    if k == 1: return np.rot90(X, 1, axes=(1, 2)), np.rot90(y, 1)
    if k == 2: return np.rot90(X, 2, axes=(1, 2)), np.rot90(y, 2)
    if k == 3: return np.rot90(X, 3, axes=(1, 2)), np.rot90(y, 3)
    if k == 4: return X[:, ::-1, :],               y[::-1, :]
    if k == 5: return X[:, :, ::-1],               y[:, ::-1]
    if k == 6: return np.rot90(X[:, ::-1, :], 1, axes=(1, 2)), np.rot90(y[::-1, :], 1)
    if k == 7: return np.rot90(X[:, :, ::-1], 1, axes=(1, 2)), np.rot90(y[:, ::-1], 1)

print('Augmentation functions defined.')
print('  8 augmentations per flood patch (4 rotations x 2 flips).')

Augmentation functions defined.
  8 augmentations per flood patch (4 rotations x 2 flips).


---
## Step 6: Extract and Save Patches

Patches are read directly from the large GeoTIFF using windowed reading
(no full file loaded into RAM) — same approach used throughout this
rebuild. This is the most time-consuming step given the larger 4-tile
extent; expect this to take longer than the original single-tile run.

In [12]:
print('Starting patch extraction...')

with rasterio.open(STACK_PATH) as src:
    for split_name, positions in splits.items():
        split_dir = os.path.join(PATCH_DIR, split_name)
        saved = 0
        flood_saved = 0
        nonflood_saved = 0

        for idx, (row, col, is_flood) in enumerate(positions):
            window = Window(col, row, PATCH_SIZE, PATCH_SIZE)
            X = src.read(window=window).astype(np.float32)  # (12, 128, 128)
            y = label_full[row:row+PATCH_SIZE, col:col+PATCH_SIZE]

            n_augs = 8 if is_flood else 1
            for k in range(n_augs):
                Xa, ya = augment(X.copy(), y.copy(), k)
                np.save(os.path.join(split_dir, f'X_{saved:05d}.npy'), Xa)
                np.save(os.path.join(split_dir, f'y_{saved:05d}.npy'), ya)
                saved += 1

            if is_flood:
                flood_saved += 1
            else:
                nonflood_saved += 1

            if (idx + 1) % 100 == 0:
                print(f'  {split_name}: {idx+1}/{len(positions)} positions  '
                      f'({saved} patches saved so far)')

        gc.collect()
        print(f'  {split_name} COMPLETE: {saved} total patches  '
              f'(flood={flood_saved}x8={flood_saved*8}  '
              f'non-flood={nonflood_saved})')
        print()

del label_full
gc.collect()
print('Patch extraction complete.')

Starting patch extraction...
  train: 100/7861 positions  (163 patches saved so far)
  train: 200/7861 positions  (347 patches saved so far)
  train: 300/7861 positions  (517 patches saved so far)
  train: 400/7861 positions  (659 patches saved so far)
  train: 500/7861 positions  (815 patches saved so far)
  train: 600/7861 positions  (950 patches saved so far)
  train: 700/7861 positions  (1127 patches saved so far)
  train: 800/7861 positions  (1311 patches saved so far)
  train: 900/7861 positions  (1460 patches saved so far)
  train: 1000/7861 positions  (1602 patches saved so far)
  train: 1100/7861 positions  (1765 patches saved so far)
  train: 1200/7861 positions  (1914 patches saved so far)
  train: 1300/7861 positions  (2056 patches saved so far)
  train: 1400/7861 positions  (2198 patches saved so far)
  train: 1500/7861 positions  (2333 patches saved so far)
  train: 1600/7861 positions  (2503 patches saved so far)
  train: 1700/7861 positions  (2666 patches saved so far)


---
## Step 7: Verify Patch Counts

In [13]:
print('=' * 60)
print('4-TILE PATCH EXTRACTION COMPLETE — SUMMARY')
print('=' * 60)
print(f'  Stack       : flood_stack_4tile_v1.tif')
print(f'  Label       : flood_label_mosaic.tif')
print(f'  Tiles       : 36JTM, 36JUM, 36JTN, 36JUN')
print(f'  Mosaic shape: {H} x {W} pixels')
print(f'  Patch size  : {PATCH_SIZE} x {PATCH_SIZE} pixels')
print(f'  Input bands : 12')
print()

total = 0
for split_name in ['train', 'val', 'test']:
    split_dir = os.path.join(PATCH_DIR, split_name)
    n = len([f for f in os.listdir(split_dir) if f.startswith('X_')])
    total += n
    print(f'  {split_name:<6} : {n:,} patches')

print(f'  Total  : {total:,} patches')
print()

4-TILE PATCH EXTRACTION COMPLETE — SUMMARY
  Stack       : flood_stack_4tile_v1.tif
  Label       : flood_label_mosaic.tif
  Tiles       : 36JTM, 36JUM, 36JTN, 36JUN
  Mosaic shape: 22139 x 21576 pixels
  Patch size  : 128 x 128 pixels
  Input bands : 12

  train  : 12,789 patches
  val    : 2,692 patches
  test   : 2,897 patches
  Total  : 18,378 patches

